# 17 — Process re-engineering

The last two days built a loop and then every layer that sits on it. This chapter asks the question those layers cannot answer: **should this process be an agent at all, and how far should it go?**

Module 01 planted three stops — `suggest`, `draft`, `act`. You now have the evidence behind them. Complexity is a cost you pay every day it is in production. Add a rung only when the one below it has actually failed you.

No model is required. If the key is down, this notebook still runs.


## 1. Learn

```
01  who decides the next step; suggest / draft / act
13  a second agent has to earn its keep
15  task, tools, grounded, cost
16  which layer you are buying
17  you are here — what should we actually automate
```

**Find the simplest thing that works, then stop.**

```mermaid
flowchart TD
    A["1. one model call"] --> B["2. plus retrieval"]
    B --> C["3. a fixed workflow"]
    C --> D["4. an agent"]
    D --> E["5. multiple agents"]
```

Most applications should stop at 2. Most production systems should stop at 3. An agent is only for a path you cannot write down in advance. Multiple agents need one of the four reasons from module 13 (different tools, permissions, model, or scale).

**Five questions, in this order.** Four of them end in "no", and that is the correct ratio.

| # | Question | If the answer is no |
|---|---|---|
| 1 | Can you write the steps down? | Build a workflow. Cheaper, testable, never a surprise. |
| 2 | Is the context written down anywhere a tool could read it? | Stop. Fix the data or the process first. |
| 3 | Is a wrong step recoverable? | It cannot sit above `draft`. Put a person on the gate. |
| 4 | Does the volume justify the build and the running cost? | Twelve cases a month is a person with a checklist. |
| 5 | Will anyone own it in six months? | If yes, and you passed 1–4, build the lowest rung that works. |

An honest no, argued from these five, is a better deliverable than a proof of concept nobody adopts.

The same three tests from module 01, now with two days behind them:

- **Creates value:** real judgment, written-down context, recoverable mistakes.
- **Still disappoints:** high volume with no tolerance, a wrong step that cannot be undone, or knowledge that lives only in people's heads. Two of those three are not AI problems.

How far, per step, not for the whole process:

- `suggest` — it recommends. A human does the thing.
- `draft` — it writes the action and waits.
- `act` — it does the thing. If you let it act, log what it did.

Cut first if the room is behind: the inbox challenge still runs. The group workshop on their own process is the thing not to cut.


## 2. Do

### Score a process you already know

Chinook, damaged media. The customer has a warped vinyl record. The policies and the ticket already exist in `data/corpus/` (`policy_damaged_media.md`, `policy_refunds.md`, ticket 06). We are not retrieving them today. We are asking whether each **step** should be a model at all.


In [1]:
refund = [
    {
        "step": "Confirm the order exists and is inside the return window.",
        "written_down": True,
        "wrong_step": "Tell them no when the policy says yes, or the reverse.",
        "autonomy": "act",
        "why": "The window is a number in a policy. A script can read it.",
    },
    {
        "step": "Decide damaged-media vs refund clock (5 to 10 business days).",
        "written_down": True,
        "wrong_step": "Quote the wrong clock. Fixable with an email.",
        "autonomy": "draft",
        "why": "Two files. Module 12 needed a second retrieve. A person sends it.",
    },
    {
        "step": "Issue the refund on the original payment method.",
        "written_down": True,
        "wrong_step": "Money leaves. Unbounded relative to a $12 CD.",
        "autonomy": "draft",
        "why": "Recoverable only as a finance mess. A person clicks.",
    },
    {
        "step": "Same 30-day reminder to every overdue invoice.",
        "written_down": True,
        "wrong_step": "A redundant email. Cheap to undo.",
        "autonomy": "act",
        "why": "You can write the steps on a slide. Module 01 row 1.",
    },
    {
        "step": "Investigate why Q3 revenue is down.",
        "written_down": False,
        "wrong_step": "A confident wrong cause becomes a wrong decision.",
        "autonomy": "suggest",
        "why": "The next query depends on the last one. A person still decides.",
    },
]

print(f"{'autonomy':8} {'ctx':5} step")
print("-" * 88)
for row in refund:
    ctx = "yes" if row["written_down"] else "no"
    print(f"{row['autonomy']:8} {ctx:5} {row['step']}")
    print(f"{'':8} {'':5} {row['why']}")


autonomy ctx   step
----------------------------------------------------------------------------------------
act      yes   Confirm the order exists and is inside the return window.
               The window is a number in a policy. A script can read it.
draft    yes   Decide damaged-media vs refund clock (5 to 10 business days).
               Two files. Module 12 needed a second retrieve. A person sends it.
draft    yes   Issue the refund on the original payment method.
               Recoverable only as a finance mess. A person clicks.
act      yes   Same 30-day reminder to every overdue invoice.
               You can write the steps on a slide. Module 01 row 1.
suggest  no    Investigate why Q3 revenue is down.
               The next query depends on the last one. A person still decides.


## 3. Observe

Three things the table already decided, before anyone opened a framework.


In [2]:
acts = [r["step"] for r in refund if r["autonomy"] == "act"]
drafts = [r["step"] for r in refund if r["autonomy"] == "draft"]
suggests = [r["step"] for r in refund if r["autonomy"] == "suggest"]
missing = [r["step"] for r in refund if not r["written_down"]]

print("act     ", len(acts), "— known steps, cheap to undo")
for s in acts:
    print("        ", s)
print("draft   ", len(drafts), "— a person on the gate, including money")
for s in drafts:
    print("        ", s)
print("suggest ", len(suggests), "— judgment, missing context, or both")
for s in suggests:
    print("        ", s)
print("no file ", len(missing), "— question 2 failed. Do not retrieve what nobody stored.")
for s in missing:
    print("        ", s)

print()
print("None of these steps needed a second agent.")
print("The refund is not act. That is the sentence for risk.")


act      2 — known steps, cheap to undo
         Confirm the order exists and is inside the return window.
         Same 30-day reminder to every overdue invoice.
draft    2 — a person on the gate, including money
         Decide damaged-media vs refund clock (5 to 10 business days).
         Issue the refund on the original payment method.
suggest  1 — judgment, missing context, or both
         Investigate why Q3 revenue is down.
no file  1 — question 2 failed. Do not retrieve what nobody stored.
         Investigate why Q3 revenue is down.

None of these steps needed a second agent.
The refund is not act. That is the sentence for risk.


## 4. Challenge

Score this support inbox. Same three words: `suggest`, `draft`, `act`. Bind `inbox`, a list of dicts with `step` and `autonomy`. Keep the step strings exactly.

The assert checks the shape, refuses `act` on the refund, and expects `act` on the reminder. The investigation must stay on `suggest`. Disagreement on the classifier is fine as long as the value is one of the three words.

After this cell, the instructor runs the group workshop on a process from *your* function. That is the artifact you take back. This inbox is the rehearsal.

| step | What you are scoring |
|---|---|
| Classify the email (billing / shipping / product). | Stable labels. Module 01 row 8. |
| Look up the customer in Chinook. | A named tool. The facts exist. |
| Issue a refund over $100. | Money. Module 01 row 7. |
| Send the same 30-day reminder template. | Known steps, every night. |
| Investigate why Q3 revenue is down. | The next query depends on the last one. |


In [ ]:
# Fill in each "autonomy" with one of: suggest, draft, act
inbox = [
    {"step": "Classify the email (billing / shipping / product).", "autonomy": ""},
    {"step": "Look up the customer in Chinook.", "autonomy": ""},
    {"step": "Issue a refund over $100.", "autonomy": ""},
    {"step": "Send the same 30-day reminder template.", "autonomy": ""},
    {"step": "Investigate why Q3 revenue is down.", "autonomy": ""},
]

In [ ]:
EXPECTED_STEPS = [
    "Classify the email (billing / shipping / product).",
    "Look up the customer in Chinook.",
    "Issue a refund over $100.",
    "Send the same 30-day reminder template.",
    "Investigate why Q3 revenue is down.",
]
allowed = {"suggest", "draft", "act"}

assert isinstance(inbox, list) and len(inbox) == 5, "inbox should be the five steps"
got_steps = [row["step"] for row in inbox]
assert got_steps == EXPECTED_STEPS, "keep the step strings exactly"
assert all(row.get("autonomy") in allowed for row in inbox), "each autonomy is suggest, draft, or act"

by_step = {row["step"]: row["autonomy"] for row in inbox}
assert by_step["Issue a refund over $100."] != "act", "money does not sit on act"
assert by_step["Send the same 30-day reminder template."] == "act", "known steps, cheap to undo"
assert by_step["Investigate why Q3 revenue is down."] == "suggest", "a wrong cause becomes a wrong decision"

print(f"{'autonomy':8} step")
print("-" * 72)
for row in inbox:
    print(f"{row['autonomy']:8} {row['step']}")


## 5. The workshop — your own process

The inbox above was the rehearsal. **This is the thing you take back.**

In groups of four, mix builders and non-builders. Pick one real process from your own function — including the parts people do informally, off the system. Then fill the table below in.

Four columns, and the fourth is the one that decides the other three:

| Column | What goes in it |
|---|---|
| `step` | One step of the real process. Include the informal ones. |
| `autonomy` | `suggest`, `draft`, or `act` — **per step**, never for the whole process. |
| `context_lives` | Where a tool would actually read this from. If the honest answer is "in Dana's head", write that. |
| `wrong_step_costs` | What it costs when this step is wrong. A correction? A customer? A regulator? |

Two rules that keep the exercise honest:

- **Any step whose context lives in someone's head cannot be automated yet.** That is question 2, and it is a data problem, not an AI problem.
- **Price the wrong step before you set the autonomy.** If you set the tier first you will talk yourself into `act`.

Concluding that your process should *not* be an agent is a valid and common outcome — and it is a better thing to take to a steering committee than another proof of concept.

In [ ]:
# Replace these rows with your own process. Add as many steps as it really has.
MY_PROCESS = [
    {
        "step": "",
        "autonomy": "",            # suggest | draft | act
        "context_lives": "",       # a system a tool could read, or "in someone's head"
        "wrong_step_costs": "",    # a correction, a customer, a regulator
    },
]


def review(rows):
    real = [r for r in rows if r["step"].strip()]
    if not real:
        print("Nothing filled in yet. Add your steps to MY_PROCESS and re-run.")
        return

    print(f"{'autonomy':9} {'context':10} step")
    print("-" * 78)
    for r in real:
        head = "IN HEAD" if "head" in r["context_lives"].lower() else "system"
        print(f"{r['autonomy'] or '?':9} {head:10} {r['step']}")

    blocked = [r for r in real if "head" in r["context_lives"].lower()]
    acting = [r for r in real if r["autonomy"] == "act"]

    print()
    print(f"steps:            {len(real)}")
    print(f"blocked on data:  {len(blocked)}  (question 2 -- fix the data first)")
    print(f"running on act:   {len(acting)}")
    for r in acting:
        print(f"   act -> {r['step']}")
        print(f"          costs: {r['wrong_step_costs'] or '(not priced -- price it)'}")
    if not acting:
        print("   nothing on act. That is a normal and defensible answer.")


review(MY_PROCESS)